# Baseline — untrained `gemma-3-1b-it` on FLORES en->sw

The control number. Scores the **stock** `google/gemma-3-1b-it` (no CPT, no SFT,
nothing trained) on the same FLORES-200 devtest, same chrF++/BLEU, same beam-8
generation you used for your fine-tuned models. This tells you how much your
whole pipeline actually added.

Reference points once you have it:
- **This baseline:** ? (what we're measuring)
- Your best SFT: 28.86 BLEU / 54.77 chrF++
- Crane swahili-gemma-1b: 27.6 / 56.8

> **Setup:** GPU **T4 ×2**, Internet ON, `HF_TOKEN` secret (Gemma is gated —
> accept its terms once on HF), FLORES+ terms accepted. ~15 min, no training.

In [5]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

CUDA_VISIBLE_DEVICES = 0


In [6]:
%%capture
!pip install -q sacrebleu bitsandbytes
!pip uninstall -y torchao

In [7]:
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("HF auth OK | CUDA:", torch.cuda.is_available())

HF auth OK | CUDA: True


## Config + prompt (identical to your training/eval format)

In [8]:
MODEL_ID = "google/gemma-3-1b-it"   # stock, untrained
MAX_LEN  = 512

def translation_prompt_infer(english):
    return ("<start_of_turn>user\n"
            "Translate the following English sentence to Swahili.\n"
            f"English: {str(english).strip()}\nSwahili:<end_of_turn>\n<start_of_turn>model\n")
print("ok")

ok


## Load the stock model (4-bit, same as training-time load)

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                         bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0},
    torch_dtype=torch.float16, attn_implementation="sdpa")
model.eval(); model.config.use_cache = True
print("loaded stock gemma-3-1b-it")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

loaded stock gemma-3-1b-it


## Load FLORES-200 devtest

In [10]:
from datasets import load_dataset
def load_flores():
    try:
        en=load_dataset("openlanguagedata/flores_plus","eng_Latn",split="devtest")
        sw=load_dataset("openlanguagedata/flores_plus","swh_Latn",split="devtest")
        return [r["text"] for r in en],[r["text"] for r in sw]
    except Exception as e:
        print("flores_plus failed (",e,") -> mirror")
        en=load_dataset("Muennighoff/flores200","eng_Latn",split="devtest")
        sw=load_dataset("Muennighoff/flores200","swh_Latn",split="devtest")
        k="sentence" if "sentence" in en.column_names else en.column_names[0]
        return [r[k] for r in en],[r[k] for r in sw]
src, ref = load_flores(); print("eval sentences:", len(src))

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/227 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/221 [00:00<?, ?it/s]

eng_Latn.jsonl: 0.00B [00:00, ?B/s]

eng_Latn.jsonl: 0.00B [00:00, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

... (4 outputs left out)

swh_Latn.jsonl: 0.00B [00:00, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

eval sentences: 1012


## Translate (beam 8) — same generation settings as your fine-tuned eval

In [11]:
tok.padding_side = "left"
def batch_translate(sentences, bs=16, max_new=256, length_penalty=1.1):
    out=[]
    for i in range(0, len(sentences), bs):
        chunk=sentences[i:i+bs]
        enc=tok([translation_prompt_infer(s) for s in chunk], return_tensors="pt",
                padding=True, truncation=True, max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            g=model.generate(**enc, max_new_tokens=max_new, num_beams=8,
                             do_sample=False, length_penalty=length_penalty, early_stopping=True)
        out+=[t.strip() for t in tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
        if i % (bs*10)==0: print(f"  {i}/{len(sentences)}")
    return out
hyp = batch_translate(src)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  0/1012
  160/1012
  320/1012
  480/1012
  640/1012
  800/1012
  960/1012


## Scorecard

In [12]:
import sacrebleu
chrf=sacrebleu.corpus_chrf(hyp,[ref],word_order=2); bleu=sacrebleu.corpus_bleu(hyp,[ref])
print("="*54)
print("  STOCK gemma-3-1b-it (untrained) en->sw")
print(f"  chrF++ : {chrf.score:5.2f}   (your SFT: 54.77 | Crane: 56.8)")
print(f"  BLEU   : {bleu.score:5.2f}   (your SFT: 28.86 | Crane: 27.6)")
print("="*54)
# show a couple of outputs — stock model is often chatty / wrong
for s,h in list(zip(src,hyp))[:3]:
    print("\nEN:", s[:80]); print("SW:", h[:80])

  STOCK gemma-3-1b-it (untrained) en->sw
  chrF++ : 14.28   (your SFT: 54.77 | Crane: 56.8)
  BLEU   :  0.41   (your SFT: 28.86 | Crane: 27.6)

EN: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," h
SW: Here are a few options for translating the sentence to Swahili, with slightly di

EN: Dr. Ehud Ur, professor of medicine at Dalhousie University in Halifax, Nova Scot
SW: Here are a few options for translating the sentence to Swahili, with slightly di

EN: Like some other experts, he is skeptical about whether diabetes can be cured, no
SW: Here are a few options for translating the sentence to Swahili, with slightly di
